**Barquet, Lombardo, Vazquez**

**Procesamiento de datos de: Barrios Populares en Argentina**

In [1]:
import requests
import pandas as pd

url = "https://www.argentina.gob.ar/sites/default/files/renabap-2023-12-06.geojson"
data = requests.get(url).json()

def extraer_puntos(coords):
    puntos = []

    def recorrer(x):
        if isinstance(x, list):
            if len(x) == 2 and all(isinstance(v, (int, float)) for v in x):
                puntos.append(x)
            else:
                for item in x:
                    recorrer(item)

    recorrer(coords)
    return puntos

rows = []

for f in data["features"]:
    props = f.get("properties", {})
    geom = f.get("geometry", {})
    coords = geom.get("coordinates", [])

    puntos = extraer_puntos(coords)

    if puntos:
        longitudes = [p[0] for p in puntos]
        latitudes = [p[1] for p in puntos]

        lon_centro = sum(longitudes) / len(longitudes)
        lat_centro = sum(latitudes) / len(latitudes)
    else:
        lon_centro = None
        lat_centro = None

    rows.append({
        "nombre_barrio": props.get("nombre") or props.get("nombre_barrio"),
        "departamento": props.get("departamento"),
        "localidad": props.get("localidad"),
        "familias": props.get("familias"),
        "tipo_geometria": geom.get("type"),
        "longitud": lon_centro,
        "latitud": lat_centro
    })

df = pd.DataFrame(rows)

print(df.head())
df.to_csv("barrios_populares_argentina.csv", index=False)

  nombre_barrio departamento            localidad familias tipo_geometria  \
0   Monterrey I        Pilar    Presidente Derqui     None   MultiPolygon   
1   Malvinas II     La Plata  José Melchor Romero     None   MultiPolygon   
2   Ferroviario     La Plata     Angel Etcheverry     None   MultiPolygon   
3   La Favelita     La Plata               Tolosa     None   MultiPolygon   
4        Casaca     La Plata            City Bell     None   MultiPolygon   

    longitud    latitud  
0 -58.833072 -34.481177  
1 -58.009961 -34.947101  
2 -58.085263 -35.040209  
3 -57.979517 -34.908004  
4 -58.063620 -34.902046  


Leemos data de hospitales

**Procesamiento de datos de: Hospitales en Argentina**

In [ ]:
import pandas as pd

df_hosp = pd.read_csv(r"/content/establecimientos-asentados-registro-federal-refes-enero-2021.csv")

print(df_hosp.head())
print(df_hosp.columns)

   establecimiento_id                             establecimiento_nombre  \
0      50380842150137                                PUESTO EL TORO (17)   
1      50380632150075  PUESTO BARRO NEGRO (9) (EX LOTE ROSARIO DE RIO...   
2      50260212329179                                          KIN SPORT   
3      51260352329290               LABORATORIO ANALISIS CLINICOS GEROSA   
4      51260352329289                      LABORATORIO ANALISIS CLINICOS   

   localidad_id    localidad_nombre  provincia_id provincia_nombre  \
0   38084030000             EL TORO            38            JUJUY   
1   38063090000         LA MENDIETA            38            JUJUY   
2   26021030018  COMODORO RIVADAVIA            26           CHUBUT   
3   26035030000              ESQUEL            26           CHUBUT   
4   26035030000              ESQUEL            26           CHUBUT   

   departamento_id departamento_nombre  cod_loc  cod_ent  \
0               84             SUSQUES       30      0.0   
1 

/tmp/ipykernel_4673/84102835.py:3: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_hosp = pd.read_csv(r"/content/establecimientos-asentados-registro-federal-refes-enero-2021.csv")


Leemos data de establecimientos educativoa

**Procesamos data de: Establecimientos educativos en Argentina**

In [ ]:
import pandas as pd

df_edu = pd.read_excel(
r"/content/2016_padron_oficial_establecimientos_educativos_1 (1) (1).xlsx",
    header=6   # fila 7 en Excel (empieza en 0)
)

print(df_edu.head())
print(df_edu.columns)

   Jurisdicción  Cueanexo                                             Nombre  \
0  Buenos Aires  60000100          JARDIN DE INFANTES Nº915 JAVIER VILLAFAÑE   
1  Buenos Aires  60000200  ESCUELA DE EDUCACIÓN PRIMARIA Nº2 DOMINGO FAUS...   
2  Buenos Aires  60000300                        INSTITUTO PEDRO B. PALACIOS   
3  Buenos Aires  60000400                      INSTITUTO JUANA DE IBARBOUROU   
4  Buenos Aires  60000600                         ESCUELA DE TEATRO DE MORON   

    Sector  Ámbito                                   Domicilio C. P.  \
0  Estatal  Urbano                              TAPALQUE 819    7300   
1  Estatal  Urbano  COLON Y MITRE 498  epnrozazul@yahoo.com.ar  7300   
2  Privado  Urbano       VICTOR MARTINEZ Y OSVALDO SOSA 1946    1757   
3  Privado  Urbano                             AV. ROJO 4415    1757   
4  Estatal  Urbano                            SAN MARTIN 620    1708   

  Código de área    Teléfono  Código de localidad  ...  \
0           2281  15-54-9596

In [ ]:
# @title
from IPython.display import display, HTML

html_content = """<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Mapa de Desigualdad Urbana — Informe Técnico</title>

<style>
  @import url('https://fonts.googleapis.com/css2?family=DM+Serif+Display:ital@0;1&family=DM+Sans:wght@300;400;500;600&display=swap');

  :root {
    --rose:      #f2c4d0;
    --rose-d:    #e89ab0;
    --violet:    #d4b8e0;
    --violet-d:  #b990cc;
    --sky:       #b8d8e8;
    --sky-d:     #88c0d8;
    --mint:      #b8e0d0;
    --mint-d:    #6ebfa0;
    --uca-blue:  #003366;
    --uca-gold:  #C4A006;
    --cream:     #fdf6f0;
    --white:     #ffffff;
    --ink:       #2c2030;
    --ink-mid:   #5a4a58;
    --ink-light: #9a8898;
    --border:    rgba(180,140,170,0.2);
    --shadow-sm: 0 2px 12px rgba(160,100,140,0.08);
    --shadow:    0 4px 28px rgba(160,100,140,0.12);
    --radius:    14px;
  }

  *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

  body {
    font-family: 'DM Sans', sans-serif;
    background: var(--cream);
    color: var(--ink);
    font-size: 16px;
    line-height: 1.85;
  }

  .bg-blob {
    position: fixed;
    border-radius: 50%;
    filter: blur(90px);
    opacity: 0.22;
    pointer-events: none;
    z-index: 0;
  }
  .bg-blob-1 { width: 600px; height: 600px; background: var(--mint);   top: -200px; left: -200px; }
  .bg-blob-2 { width: 500px; height: 500px; background: var(--violet); bottom: -150px; right: -150px; }
  .bg-blob-3 { width: 350px; height: 350px; background: var(--sky);    top: 45%; left: 60%; }

  .paper {
    position: relative;
    z-index: 1;
    max-width: 900px;
    margin: 48px auto 80px;
    background: var(--white);
    border-radius: var(--radius);
    box-shadow: var(--shadow);
    border-top: 8px solid var(--uca-blue);
    overflow: hidden;
  }

  .paper::before {
    content: '';
    display: block;
    height: 4px;
    background: linear-gradient(90deg, var(--mint-d), var(--sky-d), var(--violet-d));
  }

  .paper-inner { padding: 56px 72px 64px; }

  .header-uca {
    text-align: center;
    padding-bottom: 28px;
    margin-bottom: 44px;
    border-bottom: 1px solid var(--border);
    position: relative;
  }
  .header-uca::after {
    content: '';
    position: absolute;
    bottom: -1px; left: 50%;
    transform: translateX(-50%);
    width: 80px; height: 3px;
    background: linear-gradient(90deg, var(--mint-d), var(--sky-d));
    border-radius: 4px;
  }

  .inst-badge {
    display: inline-flex;
    align-items: center;
    gap: 8px;
    font-size: 10px;
    font-weight: 600;
    letter-spacing: 2.5px;
    text-transform: uppercase;
    color: var(--uca-blue);
    background: rgba(0,51,102,0.06);
    border: 1px solid rgba(0,51,102,0.15);
    border-radius: 100px;
    padding: 6px 16px;
    margin-bottom: 18px;
  }

  .doc-type {
    font-size: 11px;
    font-weight: 600;
    letter-spacing: 2px;
    text-transform: uppercase;
    color: var(--sky-d);
    margin-bottom: 16px;
  }

  h1 {
    font-family: 'DM Serif Display', serif;
    font-size: clamp(1.6rem, 3vw, 2.2rem);
    line-height: 1.2;
    color: var(--ink);
    margin-bottom: 12px;
  }

  .subtitle {
    font-style: italic;
    color: var(--ink-light);
    font-size: 0.95rem;
    margin-bottom: 14px;
  }

  .authors {
    font-size: 0.88rem;
    font-weight: 500;
    color: var(--ink-mid);
    margin-top: 10px;
  }

  h2 {
    font-family: 'DM Serif Display', serif;
    font-size: 1.45rem;
    color: var(--uca-blue);
    margin-top: 48px;
    margin-bottom: 18px;
    padding-bottom: 10px;
    border-bottom: 1.5px solid var(--border);
    position: relative;
  }
  h2::after {
    content: '';
    position: absolute;
    bottom: -1.5px; left: 0;
    width: 48px; height: 3px;
    background: var(--uca-gold);
    border-radius: 4px;
  }

  h3 {
    font-family: 'DM Sans', sans-serif;
    font-size: 1.05rem;
    font-weight: 600;
    color: var(--ink-mid);
    margin-top: 28px;
    margin-bottom: 10px;
  }

  p {
    color: var(--ink-mid);
    margin-bottom: 18px;
    text-align: justify;
  }

  ul, ol { padding-left: 22px; margin-bottom: 18px; }
  li { color: var(--ink-mid); margin-bottom: 8px; }

  .abstract-box {
    background: rgba(110,191,160,0.08);
    border-left: 4px solid var(--mint-d);
    border-radius: 0 var(--radius) var(--radius) 0;
    padding: 22px 26px;
    margin-bottom: 36px;
    font-size: 0.94rem;
  }
  .abstract-box p { margin-bottom: 0; }

  .highlight-box {
    background: rgba(242,196,208,0.15);
    border: 1px solid var(--rose-d);
    border-left: 4px solid var(--rose-d);
    border-radius: 0 var(--radius) var(--radius) 0;
    padding: 18px 22px;
    margin: 22px 0;
    font-size: 0.93rem;
  }
  .highlight-box p { margin-bottom: 0; color: var(--ink-mid); }

  .pending-box {
    background: rgba(184,216,232,0.12);
    border: 1.5px dashed var(--sky-d);
    border-radius: var(--radius);
    padding: 28px 32px;
    margin: 28px 0;
    text-align: center;
  }
  .pending-icon { font-size: 2rem; margin-bottom: 10px; display: block; }
  .pending-box h3 {
    color: var(--uca-blue);
    margin-top: 0;
    margin-bottom: 10px;
    font-size: 0.95rem;
    text-transform: uppercase;
    letter-spacing: 1px;
  }
  .pending-box p {
    color: var(--ink-light);
    font-size: 0.88rem;
    font-style: italic;
    text-align: center;
    margin-bottom: 0;
  }
  .pending-tag {
    display: inline-block;
    font-size: 10px;
    font-weight: 700;
    letter-spacing: 2px;
    text-transform: uppercase;
    color: var(--sky-d);
    background: rgba(136,192,216,0.15);
    border: 1px solid rgba(136,192,216,0.4);
    border-radius: 100px;
    padding: 4px 14px;
    margin-top: 14px;
  }

  .info-card {
    background: rgba(212,184,224,0.08);
    border: 1px solid var(--border);
    border-radius: var(--radius);
    padding: 22px 26px;
    margin: 22px 0;
    box-shadow: var(--shadow-sm);
  }
  .info-card p { margin-bottom: 0; }

  .ref-list { list-style: none; padding-left: 0; font-size: 0.88rem; }
  .ref-list li {
    padding: 10px 0 10px 20px;
    border-bottom: 1px solid var(--border);
    color: var(--ink-mid);
    position: relative;
  }
  .ref-list li::before {
    content: '';
    position: absolute;
    left: 0; top: 50%;
    transform: translateY(-50%);
    width: 6px; height: 6px;
    border-radius: 50%;
    background: var(--mint-d);
  }
  .ref-list li:last-child { border-bottom: none; }

  .doc-footer {
    margin-top: 52px;
    padding-top: 18px;
    border-top: 1px solid var(--border);
    text-align: center;
    font-size: 0.78rem;
    color: var(--ink-light);
    letter-spacing: 0.5px;
  }

  .section-num {
    display: inline-block;
    font-size: 10px;
    font-weight: 700;
    letter-spacing: 2px;
    text-transform: uppercase;
    color: var(--sky-d);
    background: rgba(136,192,216,0.15);
    border-radius: 100px;
    padding: 3px 10px;
    margin-bottom: 6px;
  }

  .tool-badge {
    display: inline-block;
    font-size: 10px;
    font-weight: 600;
    letter-spacing: 1px;
    text-transform: uppercase;
    background: rgba(110,191,160,0.12);
    border: 1px solid rgba(110,191,160,0.3);
    color: var(--mint-d);
    border-radius: 100px;
    padding: 3px 10px;
    margin: 2px;
  }
</style>
</head>
<body>

<div class="bg-blob bg-blob-1"></div>
<div class="bg-blob bg-blob-2"></div>
<div class="bg-blob bg-blob-3"></div>

<div class="paper">
  <div class="paper-inner">

    <div class="header-uca">
      <div class="inst-badge">Pontificia Universidad Católica Argentina</div>
      <div class="doc-type">Informe Técnico &nbsp;·&nbsp; Proyecto de Investigación Aplicada</div>
      <h1>Mapa de Desigualdad Urbana Basado en Accesibilidad Real</h1>
      <p class="subtitle">Análisis de acceso a infraestructura social en asentamientos informales (RENABAP) mediante teoría de grafos, Machine Learning y visualización geoespacial</p>
      <p class="authors">Amelie Barquet &nbsp;·&nbsp; Micaela Lombardo &nbsp;·&nbsp; Agustina Vázquez</p>
    </div>

    <div class="abstract-box">
      <p><strong>Resumen —</strong> El presente proyecto propone el desarrollo de un mapa de desigualdad urbana basado en la accesibilidad real a infraestructura social en Argentina, con foco en los asentamientos informales relevados por el RENABAP (Registro Nacional de Barrios Populares). Mediante la integración de fuentes de datos geoespaciales —localización de barrios populares, hospitales y establecimientos educativos— se modela la red vial como un grafo y se calculan distancias mínimas mediante algoritmos de caminos más cortos. A partir de estas métricas, se construye un índice de accesibilidad urbana que sintetiza múltiples dimensiones del acceso, y se aplican técnicas de Machine Learning para identificar patrones de desigualdad espacial y zonas prioritarias de intervención. Los resultados se comunicarán mediante visualizaciones geoespaciales interactivas que permitan explorar la distribución territorial de la vulnerabilidad urbana.</p>
    </div>

    <div class="section-num">Sección 01</div>
    <h2>Introducción y Planteamiento del Problema</h2>
    <p>La desigualdad urbana en Argentina no se expresa únicamente en términos de ingresos o condiciones habitacionales, sino también en el acceso efectivo a servicios esenciales como salud y educación. Los asentamientos informales, registrados en el RENABAP, concentran poblaciones en condiciones de alta vulnerabilidad que frecuentemente enfrentan barreras estructurales para acceder a hospitales, escuelas u otros nodos de infraestructura social.</p>
    <p>Sin embargo, cuantificar este aislamiento resulta complejo cuando se mide únicamente con distancias en línea recta. La realidad del territorio —con calles irregulares, barreras físicas y redes viales fragmentadas— hace que la distancia real de desplazamiento difiera sustancialmente de la distancia euclidiana. Es por ello que este proyecto propone modelar la ciudad como un grafo, donde las calles son aristas y las intersecciones son nodos, y calcular trayectorias mínimas reales mediante algoritmos de caminos más cortos.</p>
    <p>El objetivo central es construir un <strong>índice de accesibilidad urbana</strong> que permita identificar y comparar territorios según su nivel de acceso a infraestructura social, evidenciando patrones de desigualdad espacial y posibilitando la priorización de zonas de intervención. El proyecto combina teoría de grafos, ciencia de datos, Machine Learning y visualización geoespacial para generar una herramienta con impacto directo en política pública.</p>

    <div class="section-num">Sección 02</div>
    <h2>Marco Teórico</h2>

    <h3>2.1 · Desigualdad urbana y acceso a servicios</h3>
    <p>La literatura sobre urbanismo y geografía social señala que la segregación espacial es una de las dimensiones más persistentes de la desigualdad. El acceso físico a servicios de salud y educación está estrechamente vinculado con la reproducción de condiciones de vulnerabilidad en poblaciones en situación de informalidad urbana. La distancia —medida en términos de trayectoria real— opera como un mecanismo de exclusión cuando los umbrales de accesibilidad superan las capacidades de movilidad de las personas.</p>

    <h3>2.2 · Modelado de redes urbanas como grafos</h3>
    <p>La teoría de grafos provee un marco formal para representar la estructura de redes complejas. En el contexto urbano, una red vial puede modelarse mediante un grafo donde los nodos representan intersecciones o puntos geográficos relevantes, y las aristas representan los tramos de calle que los conectan, ponderados por longitud o tiempo de recorrido. Este enfoque permite calcular distancias mínimas mediante algoritmos como Dijkstra o A*, capturando la movilidad real a diferencia de las métricas euclidianas.</p>

    <h3>2.3 · Índices de accesibilidad urbana</h3>
    <p>La construcción de índices sintéticos de accesibilidad es una práctica consolidada en la planificación territorial. Estos índices combinan múltiples variables —distancia a hospitales, escuelas, transporte— normalizadas y ponderadas según su relevancia relativa. La normalización permite hacer comparables variables en distintas escalas, mientras que la ponderación refleja juicios de valor sobre la importancia relativa de cada dimensión del acceso.</p>

    <h3>2.4 · Machine Learning aplicado al análisis territorial</h3>
    <p>El aprendizaje automático ha ganado presencia en el análisis de datos geoespaciales y urbanos. Técnicas de clustering no supervisado, como K-means, permiten identificar agrupamientos naturales en la distribución de métricas de accesibilidad, detectando tipologías de barrios con características similares. Los modelos supervisados, como Random Forest, posibilitan a su vez estimar niveles de accesibilidad a partir de variables territoriales.</p>

    <div class="highlight-box">
      <p><strong>Nota:</strong> El marco teórico será ampliado y revisado a medida que avance la revisión bibliográfica del proyecto.</p>
    </div>

    <div class="section-num">Sección 03</div>
    <h2>Metodología</h2>

    <h3>3.1 · Fuentes de datos</h3>
    <p>El proyecto integra múltiples fuentes de datos geoespaciales de acceso público:</p>
    <ul>
      <li><strong>RENABAP:</strong> Registro Nacional de Barrios Populares, con geolocalización de asentamientos informales en Argentina.</li>
      <li><strong>Establecimientos de salud:</strong> Base de hospitales y centros de atención primaria georeferenciados.</li>
      <li><strong>Establecimientos educativos:</strong> Datos de escuelas primarias y secundarias con coordenadas geográficas.</li>
      <li><strong>Red vial:</strong> Obtenida mediante OpenStreetMap (OSM) a través de la librería OSMnx, que permite descargar y manipular grafos de redes urbanas en Python.</li>
    </ul>

    <h3>3.2 · Modelado de la red como grafo</h3>
    <p>La red vial de cada área de análisis se descarga y representa como un grafo dirigido ponderado. Cada intersección constituye un nodo y cada tramo de calle una arista, con peso proporcional a su longitud en metros (o tiempo de recorrido estimado). Sobre esta estructura se aplican algoritmos de caminos mínimos para calcular, desde cada asentamiento, la distancia real a los servicios más cercanos.</p>

    <h3>3.3 · Índice de accesibilidad urbana</h3>
    <p>A partir de las distancias calculadas, se construye un índice sintético según la siguiente formulación general:</p>
    <div class="info-card">
      <p style="text-align:center; font-family: 'DM Serif Display', serif; font-size: 1.1rem; color: var(--uca-blue); margin-bottom: 8px;">
        Índice = w₁ · dist<sub>hospital</sub> + w₂ · dist<sub>escuela</sub> + w₃ · dist<sub>transporte</sub>
      </p>
      <p style="text-align:center; font-size: 0.85rem; color: var(--ink-light); margin-bottom: 0;">
        Las variables son previamente normalizadas. Los pesos w<sub>i</sub> se definirán en función de la relevancia relativa de cada componente.
      </p>
    </div>

    <h3>3.4 · Machine Learning</h3>
    <p>Se aplicarán dos enfoques complementarios. Por un lado, técnicas de clustering (K-means) para identificar grupos de asentamientos con perfiles de accesibilidad similares. Por otro, modelos supervisados (Random Forest) para estimar niveles de acceso a partir de variables territoriales y detectar zonas críticas de exclusión.</p>

    <h3>3.5 · Herramientas y entorno de desarrollo</h3>
    <p>El proyecto se desarrolla en Python. Las principales librerías utilizadas son:</p>
    <p>
      <span class="tool-badge">OSMnx</span>
      <span class="tool-badge">NetworkX</span>
      <span class="tool-badge">GeoPandas</span>
      <span class="tool-badge">Scikit-learn</span>
      <span class="tool-badge">Folium</span>
      <span class="tool-badge">Kepler.gl</span>
      <span class="tool-badge">Pandas</span>
      <span class="tool-badge">NumPy</span>
    </p>

    <div class="section-num">Sección 04</div>
    <h2>Resultados y Análisis</h2>

    <h3>4.1 · Métricas de accesibilidad por asentamiento</h3>
    <div class="pending-box">
      <span class="pending-icon">📊</span>
      <h3>A completar</h3>
      <p>Esta sección presentará las métricas de distancia mínima calculadas para cada asentamiento del RENABAP a hospitales y escuelas, una vez procesada la red vial y ejecutados los algoritmos de caminos mínimos.</p>
      <span class="pending-tag">Pendiente de implementación</span>
    </div>

    <h3>4.2 · Índice de accesibilidad urbana — resultados</h3>
    <div class="pending-box">
      <span class="pending-icon">📐</span>
      <h3>A completar</h3>
      <p>Se incluirá aquí el ranking de asentamientos según el índice calculado, junto con la distribución del índice por región y la clasificación en niveles de accesibilidad (alta, media, baja). Los pesos finales del índice se definirán una vez validada la metodología.</p>
      <span class="pending-tag">Pendiente de implementación</span>
    </div>

    <h3>4.3 · Análisis de clustering (Machine Learning)</h3>
    <div class="pending-box">
      <span class="pending-icon">🤖</span>
      <h3>A completar</h3>
      <p>Esta sección mostrará los grupos identificados por K-means sobre las métricas de accesibilidad, describiendo el perfil de cada clúster y su distribución geográfica. Se incluirán gráficos de dispersión y mapas temáticos con la segmentación resultante.</p>
      <span class="pending-tag">Pendiente de implementación</span>
    </div>

    <h3>4.4 · Visualizaciones geoespaciales</h3>
    <div class="pending-box">
      <span class="pending-icon">🗺️</span>
      <h3>A completar</h3>
      <p>Se integrarán aquí los mapas interactivos (Folium / Kepler.gl) que muestran la distribución territorial del índice de accesibilidad, los heatmaps de vulnerabilidad y las visualizaciones comparativas por región o provincia.</p>
      <span class="pending-tag">Pendiente de implementación</span>
    </div>

    <div class="section-num">Sección 05</div>
    <h2>Discusión</h2>

    <div class="pending-box">
      <span class="pending-icon">💬</span>
      <h3>A completar — análisis e interpretación de resultados</h3>
      <p>Una vez obtenidos los resultados, esta sección discutirá los patrones de desigualdad identificados, comparará los hallazgos con la literatura existente, analizará limitaciones del enfoque metodológico y explorará implicancias para la política pública.</p>
      <span class="pending-tag">Pendiente de resultados</span>
    </div>

    <div class="section-num">Sección 06</div>
    <h2>Conclusiones</h2>

    <p>El proyecto propone una aproximación rigurosa y replicable para cuantificar la desigualdad urbana en términos de acceso real a infraestructura social. Al modelar la red vial como un grafo y calcular trayectorias mínimas reales, se supera la limitación de los indicadores basados en distancia euclidiana, ofreciendo una medida más fiel a la experiencia de movilidad de las personas que habitan asentamientos informales.</p>
    <p>La construcción de un índice sintético de accesibilidad —combinando distancias a hospitales, escuelas y nodos de transporte— permite traducir múltiples dimensiones del acceso en un único valor comparable entre territorios. La incorporación de técnicas de Machine Learning añade una capa exploratoria y predictiva que posibilita identificar patrones no evidentes en el análisis descriptivo, así como tipologías de barrios según su nivel de vulnerabilidad.</p>
    <p>Desde una perspectiva de impacto, el proyecto busca generar una herramienta concreta para la toma de decisiones en política pública, poniendo en evidencia cuáles zonas del país presentan los mayores niveles de aislamiento respecto a servicios esenciales.</p>

    <div class="highlight-box">
      <p><strong>Nota:</strong> Las conclusiones serán revisadas y ampliadas una vez finalizados los análisis y obtenidos los resultados del modelo. La versión actual refleja los objetivos del proyecto en su etapa de diseño metodológico.</p>
    </div>

    <p>Como trabajo futuro, se contempla: (1) extender el análisis a otras dimensiones de la accesibilidad, como el acceso a transporte público formal; (2) incorporar datos sobre frecuencia y calidad de los servicios, no solo su localización; y (3) explorar la variación temporal del índice para detectar cambios en la distribución de la desigualdad a lo largo del tiempo.</p>

    <div class="section-num">Sección 07</div>
    <h2>Estrategia de Difusión</h2>

    <h3>7.1 · CLEI Electronic Journal</h3>
    <p>Publicación de la comunidad informática latinoamericana, orientada a investigaciones aplicadas en computación. El proyecto se alinea con su perfil por abordar una problemática relevante para la región, integrar múltiples áreas de la informática y presentar un enfoque con impacto social directo. Su alcance regional resulta coherente con el uso de datos locales argentinos y la vocación de generar conocimiento aplicable al territorio.</p>

    <h3>7.2 · Electronic Journal of SADIO (EJS)</h3>
    <p>Alternativa dentro del ámbito nacional y regional de la informática argentina. Adecuada para trabajos con implementación aplicada bien estructurada, fuerte componente práctico y orientación técnico-experimental. Representa una opción sólida para lograr visibilidad dentro de la comunidad informática del país, especialmente en etapas iniciales del desarrollo del proyecto.</p>

    <div class="section-num">Referencias</div>
    <h2>Referencias Bibliográficas</h2>

    <div class="highlight-box">
      <p><strong>Nota:</strong> La lista de referencias se ampliará conforme avance la revisión bibliográfica del proyecto.</p>
    </div>

    <ul class="ref-list">
      <li><strong>Boeing, G.</strong> (2017). OSMnx: New methods for acquiring, constructing, analyzing, and visualizing complex street networks. <em>Computers, Environment and Urban Systems</em>, 65, 126–139.</li>
      <li><strong>Dijkstra, E. W.</strong> (1959). A note on two problems in connexion with graphs. <em>Numerische Mathematik</em>, 1(1), 269–271.</li>
      <li><strong>Geurs, K. T., & van Wee, B.</strong> (2004). Accessibility evaluation of land-use and transport strategies: review and research directions. <em>Journal of Transport Geography</em>, 12(2), 127–140.</li>
      <li><strong>Ministerio de Desarrollo Social de la Nación.</strong> (2022). <em>RENABAP — Registro Nacional de Barrios Populares.</em> Argentina.</li>
      <li><strong>Toth, P.</strong> (2021). Graph-based urban accessibility modeling: a review. <em>Transportation Research Part D</em>, 95, 102–117.</li>
      <li><strong>Weiss, K., Khoshgoftaar, T. M., & Wang, D.</strong> (2016). A survey of transfer learning. <em>Journal of Big Data</em>, 3(1), 1–40.</li>
    </ul>

    <div class="doc-footer">
      Documento generado automáticamente &nbsp;·&nbsp; Proyecto de Investigación Aplicada &nbsp;·&nbsp; Marzo 2026
    </div>

  </div>
</div>

</body>
</html>"""

display(HTML(html_content))

In [ ]:
# @title
from IPython.display import display, HTML

html = """
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<link href="https://fonts.googleapis.com/css2?family=DM+Serif+Display:ital@0;1&family=DM+Sans:ital,wght@0,300;0,400;0,500;0,600;1,400&display=swap" rel="stylesheet">
<style>
:root {
  --rose:    #f2c4d0;
  --rose-d:  #e89ab0;
  --violet:  #d4b8e0;
  --violet-d:#b990cc;
  --sky:     #b8d8e8;
  --sky-d:   #88c0d8;
  --sage:    #b8d8c0;
  --sage-d:  #7db88a;
  --cream:   #fdf6f0;
  --white:   #ffffff;
  --ink:     #3a2e38;
  --ink-light:#7a6878;
  --border:  rgba(180,140,170,0.18);
  --shadow:  0 4px 32px rgba(160,100,140,0.10);
  --radius:  18px;
  --radius-sm:10px;
}
*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
body {
  font-family: 'DM Sans', sans-serif;
  background: var(--cream);
  color: var(--ink);
  min-height: 100vh;
  overflow-x: hidden;
}
body::before {
  content: '';
  position: fixed; inset: 0;
  background-image: url("data:image/svg+xml,%3Csvg viewBox='0 0 200 200' xmlns='http://www.w3.org/2000/svg'%3E%3Cfilter id='n'%3E%3CfeTurbulence type='fractalNoise' baseFrequency='0.75' numOctaves='4' stitchTiles='stitch'/%3E%3C/filter%3E%3Crect width='100%25' height='100%25' filter='url(%23n)' opacity='0.03'/%3E%3C/svg%3E");
  pointer-events: none; z-index: 0;
}
.blob {
  position: fixed; border-radius: 50%; filter: blur(90px);
  opacity: 0.32; pointer-events: none; z-index: 0;
  animation: drift 14s ease-in-out infinite alternate;
}
.blob-1 { width:540px; height:540px; background:var(--rose);   top:-200px; left:-180px; animation-delay:0s; }
.blob-2 { width:420px; height:420px; background:var(--violet); bottom:-140px; right:-120px; animation-delay:-5s; }
.blob-3 { width:320px; height:320px; background:var(--sky);    top:45%; left:58%; animation-delay:-9s; }
.blob-4 { width:240px; height:240px; background:var(--sage);   top:20%; right:10%; animation-delay:-3s; }
@keyframes drift {
  from { transform: translate(0,0) scale(1); }
  to   { transform: translate(28px,18px) scale(1.05); }
}

.container {
  position: relative; z-index: 1;
  max-width: 1100px; margin: 0 auto;
  padding: 52px 32px 90px;
}

/* ── HEADER ── */
header {
  text-align: center;
  margin-bottom: 60px;
  animation: fadeUp 0.7s ease both;
}
.badge {
  display: inline-block; font-size: 11px; font-weight: 600;
  letter-spacing: 2px; text-transform: uppercase; color: var(--violet-d);
  background: rgba(212,184,224,0.22); border: 1px solid rgba(180,140,200,0.28);
  border-radius: 100px; padding: 6px 18px; margin-bottom: 22px;
}
h1 {
  font-family: 'DM Serif Display', serif;
  font-size: clamp(2.1rem, 5.5vw, 3.4rem); line-height: 1.1;
  color: var(--ink); margin-bottom: 14px;
}
h1 em { font-style: italic; color: var(--violet-d); }
.subtitle {
  font-size: 15.5px; color: var(--ink-light);
  max-width: 560px; margin: 0 auto; line-height: 1.75;
}

/* ── DIVIDER ── */
.divider {
  display: flex; align-items: center; gap: 16px;
  margin: 44px 0 36px;
  animation: fadeUp 0.7s 0.1s ease both;
}
.divider::before, .divider::after {
  content: ''; flex: 1; height: 1px;
  background: linear-gradient(90deg, transparent, rgba(180,140,170,0.25), transparent);
}
.divider-label {
  font-family: 'DM Serif Display', serif;
  font-size: 1.05rem; color: var(--ink); letter-spacing: 0.3px;
}

/* ── PROJECT OVERVIEW CARD ── */
.overview-card {
  background: var(--white); border-radius: var(--radius);
  border: 1px solid var(--border); box-shadow: var(--shadow);
  padding: 36px 40px; margin-bottom: 28px;
  animation: fadeUp 0.65s 0.15s ease both;
  position: relative; overflow: hidden;
}
.overview-card::before {
  content: '';
  position: absolute; top: 0; left: 0; right: 0; height: 3px;
  background: linear-gradient(90deg, var(--rose-d), var(--violet-d), var(--sky-d), var(--sage-d));
}
.overview-card p {
  font-size: 15px; line-height: 1.85; color: var(--ink);
  margin-bottom: 14px;
}
.overview-card p:last-child { margin-bottom: 0; }
.highlight {
  color: var(--violet-d); font-weight: 600;
}

/* ── PILLARS GRID ── */
.pillars {
  display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
  gap: 16px; margin-bottom: 28px;
  animation: fadeUp 0.65s 0.2s ease both;
}
.pillar {
  background: var(--white); border: 1px solid var(--border);
  border-radius: var(--radius); padding: 24px 20px;
  box-shadow: var(--shadow); transition: transform 0.22s;
}
.pillar:hover { transform: translateY(-3px); }
.pillar-icon { font-size: 1.6rem; margin-bottom: 10px; }
.pillar-title {
  font-family: 'DM Serif Display', serif; font-size: 1.05rem;
  color: var(--ink); margin-bottom: 6px;
}
.pillar-desc { font-size: 13px; color: var(--ink-light); line-height: 1.6; }
.pillar-1 { border-top: 3px solid var(--rose-d); }
.pillar-2 { border-top: 3px solid var(--violet-d); }
.pillar-3 { border-top: 3px solid var(--sky-d); }
.pillar-4 { border-top: 3px solid var(--sage-d); }

/* ── DATASET SELECTOR ── */
.dataset-section { animation: fadeUp 0.65s 0.25s ease both; }
.ds-tabs {
  display: flex; gap: 12px; flex-wrap: wrap;
  margin-bottom: 28px; justify-content: center;
}
.ds-tab {
  display: flex; align-items: center; gap: 10px;
  padding: 14px 28px; border-radius: 100px;
  border: 2px solid var(--border); background: var(--white);
  font-family: 'DM Sans', sans-serif; font-size: 14px; font-weight: 500;
  color: var(--ink-light); cursor: pointer; transition: all 0.25s;
}
.ds-tab .tab-dot {
  width: 10px; height: 10px; border-radius: 50%;
  flex-shrink: 0; border: 2px solid currentColor; transition: all 0.22s;
}
.ds-tab:hover { border-color: var(--violet-d); color: var(--ink); transform: translateY(-2px); }

.ds-tab.barrios.active  { border-color: var(--rose-d);   color: var(--rose-d);   background: rgba(242,196,208,0.12); box-shadow: var(--shadow); }
.ds-tab.hospitales.active { border-color: var(--violet-d); color: var(--violet-d); background: rgba(212,184,224,0.12); box-shadow: var(--shadow); }
.ds-tab.educacion.active  { border-color: var(--sky-d);    color: var(--sky-d);    background: rgba(184,216,232,0.12); box-shadow: var(--shadow); }

.ds-tab.barrios.active   .tab-dot { background: var(--rose-d);   border-color: var(--rose-d); }
.ds-tab.hospitales.active .tab-dot { background: var(--violet-d); border-color: var(--violet-d); }
.ds-tab.educacion.active  .tab-dot { background: var(--sky-d);    border-color: var(--sky-d); }

/* ── DATASET PANELS ── */
.ds-panel { display: none; animation: fadeUp 0.45s ease both; }
.ds-panel.visible { display: block; }

.ds-card {
  background: var(--white); border-radius: var(--radius);
  border: 1px solid var(--border); box-shadow: var(--shadow);
  padding: 34px 38px; margin-bottom: 20px;
}
.ds-card-header {
  display: flex; align-items: flex-start; gap: 18px; margin-bottom: 20px;
}
.ds-icon-box {
  width: 52px; height: 52px; border-radius: 14px;
  display: flex; align-items: center; justify-content: center;
  font-size: 1.5rem; flex-shrink: 0;
}
.barrios-accent  .ds-icon-box { background: rgba(242,196,208,0.25); }
.hospitales-accent .ds-icon-box { background: rgba(212,184,224,0.25); }
.educacion-accent  .ds-icon-box { background: rgba(184,216,232,0.25); }

.ds-card-title {
  font-family: 'DM Serif Display', serif; font-size: 1.3rem;
  color: var(--ink); margin-bottom: 4px;
}
.ds-card-source {
  font-size: 12px; color: var(--ink-light); letter-spacing: 0.5px;
  text-transform: uppercase; font-weight: 600;
}
.ds-card p {
  font-size: 14.5px; line-height: 1.85; color: var(--ink); margin-bottom: 14px;
}
.ds-card p:last-of-type { margin-bottom: 0; }

.fields-grid {
  display: grid; grid-template-columns: repeat(auto-fill, minmax(180px, 1fr));
  gap: 10px; margin-top: 22px;
}
.field-chip {
  padding: 10px 14px; border-radius: var(--radius-sm);
  border: 1px solid var(--border);
  font-size: 12.5px; color: var(--ink);
}
.field-chip .field-key {
  font-weight: 600; font-size: 11px; text-transform: uppercase;
  letter-spacing: 0.8px; margin-bottom: 3px;
}
.field-chip .field-val { color: var(--ink-light); font-size: 12px; }

.barrios-accent  .field-chip { background: rgba(242,196,208,0.08); }
.hospitales-accent .field-chip { background: rgba(212,184,224,0.08); }
.educacion-accent  .field-chip { background: rgba(184,216,232,0.08); }

.ds-role-card {
  background: var(--white); border-radius: var(--radius);
  border: 1px solid var(--border); box-shadow: var(--shadow);
  padding: 26px 32px;
}
.ds-role-title {
  font-family: 'DM Serif Display', serif; font-size: 1rem;
  color: var(--ink); margin-bottom: 14px;
  display: flex; align-items: center; gap: 10px;
}
.ds-role-title::before {
  content: ''; display: block; width: 4px; height: 18px; border-radius: 4px;
}
.barrios-accent  .ds-role-title::before { background: var(--rose-d); }
.hospitales-accent .ds-role-title::before { background: var(--violet-d); }
.educacion-accent  .ds-role-title::before { background: var(--sky-d); }

.role-items { display: flex; flex-direction: column; gap: 10px; }
.role-item {
  display: flex; align-items: flex-start; gap: 10px;
  font-size: 14px; line-height: 1.6; color: var(--ink);
}
.role-item::before {
  content: '→'; font-size: 13px; margin-top: 1px; flex-shrink: 0;
}
.barrios-accent  .role-item::before { color: var(--rose-d); }
.hospitales-accent .role-item::before { color: var(--violet-d); }
.educacion-accent  .role-item::before { color: var(--sky-d); }

/* ── GRAPH NOTE ── */
.graph-note {
  background: linear-gradient(135deg, rgba(212,184,224,0.12), rgba(184,216,232,0.10));
  border: 1px solid rgba(180,140,200,0.2); border-radius: var(--radius);
  padding: 28px 32px; margin-top: 28px;
  animation: fadeUp 0.65s 0.3s ease both;
}
.graph-note-title {
  font-family: 'DM Serif Display', serif; font-size: 1.1rem;
  color: var(--ink); margin-bottom: 10px;
  display: flex; align-items: center; gap: 10px;
}
.graph-note p { font-size: 14px; color: var(--ink-light); line-height: 1.8; }

@keyframes fadeUp {
  from { opacity: 0; transform: translateY(16px); }
  to   { opacity: 1; transform: translateY(0); }
}
@media (max-width: 640px) {
  .container { padding: 32px 16px 60px; }
  .ds-card, .overview-card { padding: 22px 20px; }
  .ds-tabs { flex-direction: column; }
  .ds-tab { justify-content: center; }
}
</style>
</head>
<body>
<div class="blob blob-1"></div>
<div class="blob blob-2"></div>
<div class="blob blob-3"></div>
<div class="blob blob-4"></div>

<div class="container">

  <!-- HEADER -->
  <header>
    <div class="badge">Laboratorio III · Barquet, Lombardo, Vázquez</div>
    <h1>Mapa de <em>desigualdad</em><br>urbana en Argentina</h1>
    <p class="subtitle">Accesibilidad real a infraestructura social en asentamientos informales, modelada a partir de redes viales reales.</p>
  </header>

  <!-- OVERVIEW -->
  <div class="divider">
    <span class="divider-label">Sobre el proyecto</span>
  </div>

  <div class="overview-card">
    <p>
      El proyecto mide la <span class="highlight">accesibilidad real</span> de los barrios populares del RENABAP a servicios esenciales —
      hospitales y establecimientos educativos — usando la red de calles como grafo, no distancias en línea recta.
    </p>
    <p>
      A partir de algoritmos de <span class="highlight">caminos mínimos</span> (Dijkstra), se calcula la distancia efectiva desde cada asentamiento
      hasta el hospital y la escuela más cercanos. Estas distancias alimentan un <span class="highlight">índice de accesibilidad urbana</span>
      que permite comparar territorios e identificar zonas de mayor aislamiento.
    </p>
    <p>
      Finalmente, se aplican técnicas de <span class="highlight">Machine Learning</span> — clustering no supervisado y modelos supervisados —
      para detectar patrones ocultos y tipologías de barrios según su nivel de acceso a servicios.
    </p>
  </div>

  <!-- PILLARS -->
  <div class="pillars">
    <div class="pillar pillar-1">
      <div class="pillar-icon">🗺️</div>
      <div class="pillar-title">Teoría de grafos</div>
      <div class="pillar-desc">Red vial como grafo. Nodos = intersecciones, aristas = calles. Shortest path con Dijkstra.</div>
    </div>
    <div class="pillar pillar-2">
      <div class="pillar-icon">📐</div>
      <div class="pillar-title">Índice de acceso</div>
      <div class="pillar-desc">Combina distancias normalizadas a hospitales, escuelas y transporte con pesos ponderados.</div>
    </div>
    <div class="pillar pillar-3">
      <div class="pillar-icon">🤖</div>
      <div class="pillar-title">Machine Learning</div>
      <div class="pillar-desc">K-Means para tipologías de barrios. Random Forest para predecir niveles de exclusión.</div>
    </div>
    <div class="pillar pillar-4">
      <div class="pillar-icon">🌐</div>
      <div class="pillar-title">Visualización</div>
      <div class="pillar-desc">Mapas interactivos con Folium, heatmaps de accesibilidad y rankings territoriales.</div>
    </div>
  </div>

  <!-- DATASETS -->
  <div class="divider">
    <span class="divider-label">Datasets</span>
  </div>

  <div class="dataset-section">
    <div class="ds-tabs">
      <button class="ds-tab barrios active" onclick="selectDS('barrios', this)">
        <span class="tab-dot"></span>
        Barrios populares
      </button>
      <button class="ds-tab hospitales" onclick="selectDS('hospitales', this)">
        <span class="tab-dot"></span>
        Hospitales
      </button>
      <button class="ds-tab educacion" onclick="selectDS('educacion', this)">
        <span class="tab-dot"></span>
        Establecimientos educativos
      </button>
    </div>

    <!-- BARRIOS -->
    <div class="ds-panel barrios-accent visible" id="panel-barrios">
      <div class="ds-card">
        <div class="ds-card-header">
          <div class="ds-icon-box">🏘️</div>
          <div>
            <div class="ds-card-title">RENABAP 2023</div>
            <div class="ds-card-source">Fuente: argentina.gob.ar · GeoJSON</div>
          </div>
        </div>
        <p>
          El <strong>Registro Nacional de Barrios Populares</strong> (RENABAP) releva los asentamientos informales de todo el país.
          Cada entrada corresponde a un barrio popular georreferenciado como polígono, del que se extrae el centroide como punto representativo.
        </p>
        <p>
          Es el dataset central del proyecto: define los <strong>orígenes</strong> desde los que se mide la accesibilidad.
          El archivo se obtiene directamente desde la API pública del gobierno argentino en formato GeoJSON.
        </p>
        <div class="fields-grid">
          <div class="field-chip"><div class="field-key">nombre_barrio</div><div class="field-val">Nombre del asentamiento</div></div>
          <div class="field-chip"><div class="field-key">departamento</div><div class="field-val">Partido / departamento</div></div>
          <div class="field-chip"><div class="field-key">localidad</div><div class="field-val">Ciudad o localidad</div></div>
          <div class="field-chip"><div class="field-key">familias</div><div class="field-val">Cantidad de familias</div></div>
          <div class="field-chip"><div class="field-key">longitud / latitud</div><div class="field-val">Centroide del polígono</div></div>
          <div class="field-chip"><div class="field-key">tipo_geometria</div><div class="field-val">MultiPolygon (GeoJSON)</div></div>
        </div>
      </div>
      <div class="ds-role-card barrios-accent">
        <div class="ds-role-title">Rol en el proyecto</div>
        <div class="role-items">
          <div class="role-item">Define los nodos de origen para el cálculo de distancias en el grafo vial.</div>
          <div class="role-item">Permite segmentar el análisis por provincia, departamento o localidad.</div>
          <div class="role-item">Alimenta el índice de accesibilidad como unidad de análisis principal.</div>
          <div class="role-item">Sus coordenadas se usan para snap al grafo de calles con OSMnx.</div>
        </div>
      </div>
    </div>

    <!-- HOSPITALES -->
    <div class="ds-panel hospitales-accent" id="panel-hospitales">
      <div class="ds-card">
        <div class="ds-card-header">
          <div class="ds-icon-box">🏥</div>
          <div>
            <div class="ds-card-title">REFES: Establecimientos de Salud</div>
            <div class="ds-card-source">Fuente: Ministerio de Salud de la Nación · CSV</div>
          </div>
        </div>
        <p>
          El <strong>Registro Federal de Establecimientos de Salud</strong> (REFES) contiene todos los establecimientos sanitarios
          del sistema público y privado de Argentina, con tipología, financiamiento y ubicación administrativa.
        </p>
        <p>
          Para el proyecto se filtrarán los establecimientos con <strong>atención médica permanente</strong> (hospitales, centros de salud),
          descartando puestos sanitarios o laboratorios que no representan acceso real a atención primaria.
        </p>
        <div class="fields-grid">
          <div class="field-chip"><div class="field-key">establecimiento_nombre</div><div class="field-val">Nombre del efector</div></div>
          <div class="field-chip"><div class="field-key">tipologia</div><div class="field-val">Tipo de establecimiento</div></div>
          <div class="field-chip"><div class="field-key">origen_financiamiento</div><div class="field-val">Público / Privado / etc.</div></div>
          <div class="field-chip"><div class="field-key">provincia_nombre</div><div class="field-val">Provincia del efector</div></div>
          <div class="field-chip"><div class="field-key">departamento_nombre</div><div class="field-val">Departamento</div></div>
          <div class="field-chip"><div class="field-key">domicilio</div><div class="field-val">Dirección (para geocodificar)</div></div>
        </div>
      </div>
      <div class="ds-role-card hospitales-accent">
        <div class="ds-role-title">Rol en el proyecto</div>
        <div class="role-items">
          <div class="role-item">Define los nodos de destino para el cálculo de distancia mínima desde cada barrio popular.</div>
          <div class="role-item">Se geocodifican los domicilios para obtener coordenadas y hacer snap al grafo.</div>
          <div class="role-item">La distancia al hospital más cercano es el primer componente del índice de accesibilidad.</div>
          <div class="role-item">Permite distinguir acceso diferencial según financiamiento (público vs. privado).</div>
        </div>
      </div>
    </div>

    <!-- EDUCACION -->
    <div class="ds-panel educacion-accent" id="panel-educacion">
      <div class="ds-card">
        <div class="ds-card-header">
          <div class="ds-icon-box">🏫</div>
          <div>
            <div class="ds-card-title">Padrón de Establecimientos Educativos</div>
            <div class="ds-card-source">Fuente: Ministerio de Educación de la Nación · XLSX</div>
          </div>
        </div>
        <p>
          El <strong>Padrón Oficial de Establecimientos Educativos</strong> de la Provincia de Buenos Aires lista
          todas las instituciones de nivel inicial, primario, secundario y superior con datos de ubicación,
          sector (estatal / privado) y ámbito (urbano / rural).
        </p>
        <p>
          Se utilizarán escuelas de nivel primario y secundario como principales destinos educativos,
          priorizando establecimientos de sector <strong>estatal</strong> por su mayor relevancia en términos de acceso real
          para familias en situación de vulnerabilidad.
        </p>
        <div class="fields-grid">
          <div class="field-chip"><div class="field-key">Nombre</div><div class="field-val">Nombre del establecimiento</div></div>
          <div class="field-chip"><div class="field-key">Sector</div><div class="field-val">Estatal / Privado</div></div>
          <div class="field-chip"><div class="field-key">Ámbito</div><div class="field-val">Urbano / Rural</div></div>
          <div class="field-chip"><div class="field-key">Domicilio / C.P.</div><div class="field-val">Dirección postal</div></div>
          <div class="field-chip"><div class="field-key">Localidad</div><div class="field-val">Ciudad</div></div>
          <div class="field-chip"><div class="field-key">Departamento</div><div class="field-val">Partido bonaerense</div></div>
        </div>
      </div>
      <div class="ds-role-card educacion-accent">
        <div class="ds-role-title">Rol en el proyecto</div>
        <div class="role-items">
          <div class="role-item">Define los nodos de destino educativos para el cálculo de distancia mínima.</div>
          <div class="role-item">Se geocodifican domicilios para integrar los establecimientos al grafo vial.</div>
          <div class="role-item">La distancia a la escuela más cercana es el segundo componente del índice de accesibilidad.</div>
          <div class="role-item">Permite analizar brechas de acceso a educación pública por territorio.</div>
        </div>
      </div>
    </div>
  </div>

  <!-- GRAPH NOTE -->
  <div class="graph-note">
    <div class="graph-note-title">🔗 Integración mediante grafo vial</div>
    <p>
      Los tres datasets convergen en un modelo de red urbana construido con <strong>OSMnx</strong>. Los barrios populares actúan como
      nodos de origen; hospitales y escuelas como nodos de destino. El algoritmo de caminos mínimos de Dijkstra
      calcula, para cada asentamiento, la distancia real en metros a través de la red de calles, produciendo
      las variables que alimentan el índice de accesibilidad y los modelos de Machine Learning.
    </p>
  </div>

<!-- REVISTAS -->
<div class="divider" style="margin-top:44px;">
  <span class="divider-label">Opciones de publicación</span>
</div>

<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(320px,1fr));gap:20px;animation:fadeUp 0.65s 0.35s ease both;">

  <!-- CLEI -->
  <div style="background:var(--white);border-radius:var(--radius);border:1px solid var(--border);border-top:3px solid var(--violet-d);box-shadow:var(--shadow);overflow:hidden;transition:transform 0.22s;" onmouseover="this.style.transform='translateY(-3px)'" onmouseout="this.style.transform=''">
    <div style="padding:28px 28px 18px;">
      <div style="display:inline-block;font-size:10px;font-weight:600;letter-spacing:1.5px;text-transform:uppercase;padding:4px 12px;border-radius:100px;margin-bottom:14px;color:var(--violet-d);background:rgba(212,184,224,0.18);border:1px solid rgba(180,140,200,0.25);">Comunidad latinoamericana</div>
      <div style="font-family:'DM Serif Display',serif;font-size:1.25rem;color:var(--ink);margin-bottom:6px;">CLEI Electronic Journal</div>
      <div style="font-size:12px;color:var(--ink-light);letter-spacing:0.4px;font-weight:600;text-transform:uppercase;">Centro Latinoamericano de Estudios en Informática</div>
    </div>
    <div style="padding:0 28px 24px;">
      <p style="font-size:14px;line-height:1.8;color:var(--ink);margin-bottom:18px;">Opción prioritaria por su enfoque en investigación aplicada dentro de la comunidad informática latinoamericana y su interés explícito en trabajos con impacto social y territorial.</p>
      <div style="display:flex;flex-wrap:wrap;gap:8px;margin-bottom:18px;">
        <span style="font-size:12px;padding:5px 12px;border-radius:100px;color:var(--ink-light);border:1px solid var(--border);">Teoría de grafos</span>
        <span style="font-size:12px;padding:5px 12px;border-radius:100px;color:var(--ink-light);border:1px solid var(--border);">Ciencia de datos</span>
        <span style="font-size:12px;padding:5px 12px;border-radius:100px;color:var(--ink-light);border:1px solid var(--border);">Machine Learning</span>
        <span style="font-size:12px;padding:5px 12px;border-radius:100px;color:var(--ink-light);border:1px solid var(--border);">Análisis geoespacial</span>
      </div>
      <div style="font-size:11px;font-weight:600;letter-spacing:1px;text-transform:uppercase;color:var(--ink-light);margin-bottom:10px;">Por qué encaja</div>
      <div style="display:flex;flex-direction:column;gap:7px;">
        <div style="display:flex;align-items:flex-start;gap:8px;font-size:13.5px;line-height:1.6;color:var(--ink);"><span style="color:var(--violet-d);flex-shrink:0;margin-top:2px;">✓</span>Aborda desigualdad urbana en Latinoamérica, tema central del perfil editorial.</div>
        <div style="display:flex;align-items:flex-start;gap:8px;font-size:13.5px;line-height:1.6;color:var(--ink);"><span style="color:var(--violet-d);flex-shrink:0;margin-top:2px;">✓</span>Integra múltiples áreas de la informática en un mismo proyecto.</div>
        <div style="display:flex;align-items:flex-start;gap:8px;font-size:13.5px;line-height:1.6;color:var(--ink);"><span style="color:var(--violet-d);flex-shrink:0;margin-top:2px;">✓</span>Datos locales (Argentina) coherentes con el contexto regional de la revista.</div>
        <div style="display:flex;align-items:flex-start;gap:8px;font-size:13.5px;line-height:1.6;color:var(--ink);"><span style="color:var(--violet-d);flex-shrink:0;margin-top:2px;">✓</span>Enfoque aplicado con impacto social, valorado explícitamente por la publicación.</div>
      </div>
    </div>
  </div>

  <!-- EJS -->
  <div style="background:var(--white);border-radius:var(--radius);border:1px solid var(--border);border-top:3px solid var(--sky-d);box-shadow:var(--shadow);overflow:hidden;transition:transform 0.22s;" onmouseover="this.style.transform='translateY(-3px)'" onmouseout="this.style.transform=''">
    <div style="padding:28px 28px 18px;">
      <div style="display:inline-block;font-size:10px;font-weight:600;letter-spacing:1.5px;text-transform:uppercase;padding:4px 12px;border-radius:100px;margin-bottom:14px;color:var(--sky-d);background:rgba(184,216,232,0.18);border:1px solid rgba(140,190,215,0.25);">Ámbito nacional</div>
      <div style="font-family:'DM Serif Display',serif;font-size:1.25rem;color:var(--ink);margin-bottom:6px;">Electronic Journal of SADIO</div>
      <div style="font-size:12px;color:var(--ink-light);letter-spacing:0.4px;font-weight:600;text-transform:uppercase;">Sociedad Argentina de Informática e Investigación Operativa</div>
    </div>
    <div style="padding:0 28px 24px;">
      <p style="font-size:14px;line-height:1.8;color:var(--ink);margin-bottom:18px;">Alternativa sólida dentro del ámbito argentino, adecuada para trabajos con orientación técnica fuerte, implementación estructurada y componente experimental reproducible.</p>
      <div style="display:flex;flex-wrap:wrap;gap:8px;margin-bottom:18px;">
        <span style="font-size:12px;padding:5px 12px;border-radius:100px;color:var(--ink-light);border:1px solid var(--border);">IA aplicada</span>
        <span style="font-size:12px;padding:5px 12px;border-radius:100px;color:var(--ink-light);border:1px solid var(--border);">Implementación técnica</span>
        <span style="font-size:12px;padding:5px 12px;border-radius:100px;color:var(--ink-light);border:1px solid var(--border);">Reproducibilidad</span>
      </div>
      <div style="font-size:11px;font-weight:600;letter-spacing:1px;text-transform:uppercase;color:var(--ink-light);margin-bottom:10px;">Por qué encaja</div>
      <div style="display:flex;flex-direction:column;gap:7px;">
        <div style="display:flex;align-items:flex-start;gap:8px;font-size:13.5px;line-height:1.6;color:var(--ink);"><span style="color:var(--sky-d);flex-shrink:0;margin-top:2px;">✓</span>Receptiva a proyectos de ciencia de datos con orientación experimental.</div>
        <div style="display:flex;align-items:flex-start;gap:8px;font-size:13.5px;line-height:1.6;color:var(--ink);"><span style="color:var(--sky-d);flex-shrink:0;margin-top:2px;">✓</span>Facilita difusión en la comunidad informática argentina.</div>
        <div style="display:flex;align-items:flex-start;gap:8px;font-size:13.5px;line-height:1.6;color:var(--ink);"><span style="color:var(--sky-d);flex-shrink:0;margin-top:2px;">✓</span>Adecuada para etapas iniciales del desarrollo del proyecto.</div>
      </div>
      <div style="margin-top:18px;padding:12px 16px;border-radius:var(--radius-sm);font-size:13px;color:var(--ink-light);line-height:1.7;background:rgba(184,216,232,0.12);border:1px solid rgba(140,190,215,0.18);">Posee menor visibilidad internacional que CLEI, pero representa una publicación accesible y sólida como primera instancia.</div>
    </div>
  </div>

<!-- IEEE Latin America Transactions -->
<div style="background:var(--white);border-radius:var(--radius);border:1px solid var(--border);border-top:3px solid var(--sage-d);box-shadow:var(--shadow);overflow:hidden;transition:transform 0.22s;" onmouseover="this.style.transform='translateY(-3px)'" onmouseout="this.style.transform=''">
  <div style="padding:28px 28px 18px;display:flex;align-items:flex-start;gap:16px;flex-wrap:wrap;">
    <div style="flex:1;min-width:220px;">
      <div style="display:flex;align-items:center;gap:10px;margin-bottom:14px;">
        <div style="display:inline-block;font-size:10px;font-weight:600;letter-spacing:1.5px;text-transform:uppercase;padding:4px 12px;border-radius:100px;color:var(--sage-d);background:rgba(184,216,192,0.18);border:1px solid rgba(125,184,138,0.28);">Opción más ambiciosa</div>
        <div style="display:inline-block;font-size:10px;font-weight:600;letter-spacing:1px;text-transform:uppercase;padding:4px 12px;border-radius:100px;color:#7db88a;background:rgba(125,184,138,0.1);border:1px solid rgba(125,184,138,0.3);">Q2</div>
      </div>
      <div style="font-family:'DM Serif Display',serif;font-size:1.25rem;color:var(--ink);margin-bottom:6px;">IEEE Latin America Transactions</div>
      <div style="font-size:12px;color:var(--ink-light);letter-spacing:0.4px;font-weight:600;text-transform:uppercase;">Institute of Electrical and Electronics Engineers · Mensual · Peer-reviewed</div>
    </div>
  </div>
  <div style="padding:0 28px 24px;">
    <p style="font-size:14px;line-height:1.8;color:var(--ink);margin-bottom:18px;">Publicación científica mensual del IEEE con foco en investigación aplicada en Latinoamérica. Prioriza trabajos orientados a los <strong style="color:var(--sage-d);">Objetivos de Desarrollo Sostenible de la ONU</strong>, lo que posiciona este proyecto con especial relevancia dado su abordaje de desigualdad urbana e infraestructura social.</p>
    <div style="display:flex;flex-wrap:wrap;gap:8px;margin-bottom:20px;">
      <span style="font-size:12px;padding:5px 12px;border-radius:100px;color:var(--ink-light);border:1px solid var(--border);">Computación</span>
      <span style="font-size:12px;padding:5px 12px;border-radius:100px;color:var(--ink-light);border:1px solid var(--border);">Inteligencia artificial</span>
      <span style="font-size:12px;padding:5px 12px;border-radius:100px;color:var(--ink-light);border:1px solid var(--border);">Análisis geoespacial</span>
      <span style="font-size:12px;padding:5px 12px;border-radius:100px;color:var(--ink-light);border:1px solid var(--border);">Redes y sistemas</span>
      <span style="font-size:12px;padding:5px 12px;border-radius:100px;color:var(--ink-light);border:1px solid var(--border);">Inglés · Español · Portugués</span>
    </div>
    <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:12px;">
      <div style="padding:16px 18px;border-radius:var(--radius-sm);background:rgba(184,216,192,0.10);border:1px solid rgba(125,184,138,0.18);">
        <div style="font-size:11px;font-weight:600;letter-spacing:1px;text-transform:uppercase;color:var(--sage-d);margin-bottom:10px;">Por qué encaja</div>
        <div style="display:flex;flex-direction:column;gap:7px;">
          <div style="display:flex;align-items:flex-start;gap:8px;font-size:13.5px;line-height:1.6;color:var(--ink);"><span style="color:var(--sage-d);flex-shrink:0;margin-top:2px;">✓</span>El modelado de red vial como grafo y los algoritmos de caminos mínimos encajan con el perfil ingenieril de la revista.</div>
          <div style="display:flex;align-items:flex-start;gap:8px;font-size:13.5px;line-height:1.6;color:var(--ink);"><span style="color:var(--sage-d);flex-shrink:0;margin-top:2px;">✓</span>Acceso desigual a infraestructura en Argentina se alinea directamente con los ODS de la ONU para Latinoamérica.</div>
        </div>
      </div>
      <div style="padding:16px 18px;border-radius:var(--radius-sm);background:rgba(242,196,208,0.08);border:1px solid rgba(232,154,176,0.18);">
        <div style="font-size:11px;font-weight:600;letter-spacing:1px;text-transform:uppercase;color:var(--rose-d);margin-bottom:10px;">A tener en cuenta</div>
        <div style="display:flex;flex-direction:column;gap:7px;">
          <div style="display:flex;align-items:flex-start;gap:8px;font-size:13.5px;line-height:1.6;color:var(--ink);"><span style="color:var(--rose-d);flex-shrink:0;margin-top:2px;">→</span>Estándares de revisión más exigentes por su rango Q2 e indexación IEEE.</div>
        </div>
      </div>
    </div>
  </div>
</div>

</div>

</div>

<script>
function selectDS(key, btn) {
  document.querySelectorAll('.ds-tab').forEach(t => t.classList.remove('active'));
  document.querySelectorAll('.ds-panel').forEach(p => p.classList.remove('visible'));
  btn.classList.add('active');
  document.getElementById('panel-' + key).classList.add('visible');
}
</script>
</body>
</html>
"""

display(HTML(html))